# Dataset split

The corpus is divided into three parts that share no identity: **train** (85%) fits AggFace, **validation** (5%) selects its checkpoint, and **test** (10%) is what the recognition experiment scores. Keeping all three disjoint by *person* is what makes a test result a statement about unseen people rather than about memorised ones; splitting by video would leave the same face on both sides.

Validation is separate from test on purpose. A checkpoint chosen on the same identities it is later reported on is chosen partly for them, which inflates the reported number.

This notebook reads `data/voxceleb2.parquet` and writes `data/voxceleb2_split.parquet`, one row per identity.


In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from meles.paths import repository_root

ROOT = repository_root()
STRUCTURE = ROOT / "data/voxceleb2.parquet"
SPLIT = ROOT / "data/voxceleb2_split.parquet"

SEED = 0
VALIDATION, TEST = 0.05, 0.1

structure = pd.read_parquet(STRUCTURE, columns=["identity", "video"])
identities = np.sort(structure["identity"].unique())

# Split the identities themselves, so no person can appear in two parts.
train, held_out = train_test_split(identities, test_size=VALIDATION + TEST, random_state=SEED)
validation, test = train_test_split(held_out, test_size=TEST / (VALIDATION + TEST), random_state=SEED)

assigned = {
    **dict.fromkeys(train, "train"),
    **dict.fromkeys(validation, "validation"),
    **dict.fromkeys(test, "test"),
}
split = pd.DataFrame({"identity": identities, "split": [assigned[identity] for identity in identities]})
split.to_parquet(SPLIT, index=False)

split["split"].value_counts().to_frame("identities")


,identities
split,
train,5094
test,600
validation,300


## What each part holds

The parts are drawn at random, so they should look alike apart from their size. Two numbers decide whether the split is usable:

- **videos per identity** in the test part, because the experiment's k-fold split needs every test identity to hold at least k videos;
- **frames per identity** in train, because that is what AggFace has to learn from.

In [8]:
assert set(split["identity"]) == set(identities), "every identity must be assigned"
assert split["identity"].is_unique, "an identity must be assigned exactly once"

labelled = structure.merge(split, on="identity")
videos_per_identity = labelled.groupby(["split", "identity"])["video"].nunique()
frames_per_identity = labelled.groupby(["split", "identity"]).size()

summary = pd.DataFrame(
    {
        "identities": videos_per_identity.groupby("split").size(),
        "videos": videos_per_identity.groupby("split").sum(),
        "frames": frames_per_identity.groupby("split").sum(),
        "videos/identity min": videos_per_identity.groupby("split").min(),
        "videos/identity median": videos_per_identity.groupby("split").median(),
        "frames/identity median": frames_per_identity.groupby("split").median(),
    }
)
summary.loc[["train", "validation", "test"]]


,identities,videos,frames,videos/identity min,videos/identity median,frames/identity median
split,,,,,,
train,5094,124370,2905144,6,20.0,438.5
validation,300,6774,157040,6,19.0,387.0
test,600,14399,338418,6,20.0,423.0
